# 🏢 HR Analytics — Employee Attrition Prediction

**Project Goal:** Predict which employees are likely to leave and identify key attrition drivers

**Tools:** Python · Pandas · NumPy · Matplotlib · Seaborn · Scikit-learn

---

## ⚙️ Step 0: Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, accuracy_score,
                              precision_score, recall_score, f1_score)
import joblib

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print('✔ All imports successful!')

## 📥 Step 1: Data Ingestion

First, generate or load the HR dataset.

In [ ]:
# Generate dataset if not present
import subprocess
if not os.path.exists('../data/HR_Analytics.csv'):
    subprocess.run([sys.executable, '../data/generate_dataset.py'])

df = pd.read_csv('../data/HR_Analytics.csv')
print(f'Shape: {df.shape}')
print(f'Attrition rate: {(df["Attrition"]=="Yes").mean()*100:.1f}%')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().round(2)

## 🧹 Step 2: Data Cleaning & Preprocessing

In [ ]:
print('Missing values:', df.isnull().sum().sum())
print('Duplicates:', df.duplicated().sum())

drop_cols = ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber']
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)

df['Attrition_Flag'] = (df['Attrition'] == 'Yes').astype(int)
print(f'\nAttrition:\n{df["Attrition"].value_counts()}')

## 📊 Step 3: Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
counts = df['Attrition'].value_counts()
axes[0].pie(counts, labels=counts.index, autopct='%1.1f%%',
            colors=['#16A34A','#DC2626'], wedgeprops={'edgecolor':'white'})
axes[0].set_title('Attrition Distribution')

dept_attr = df.groupby('Department')['Attrition_Flag'].mean().sort_values()*100
dept_attr.plot.barh(ax=axes[1], color='#2563EB')
axes[1].set_title('Attrition Rate by Department')
axes[1].set_xlabel('Attrition Rate (%)')
plt.tight_layout()
plt.show()

In [ ]:
# Satisfaction heatmap
sat_fields = ['JobSatisfaction','EnvironmentSatisfaction','RelationshipSatisfaction','WorkLifeBalance']
sat_hm = pd.DataFrame({col: df.groupby(col)['Attrition_Flag'].mean()*100 for col in sat_fields}).T

fig, ax = plt.subplots(figsize=(9, 4))
sns.heatmap(sat_hm, annot=True, fmt='.1f', cmap='RdYlGn_r', ax=ax,
            cbar_kws={'label': 'Attrition Rate (%)'})
ax.set_title('Attrition Rate by Satisfaction Score (1=Low, 4=High)')
ax.set_xlabel('Score')
plt.tight_layout()
plt.show()

In [ ]:
# Overtime, Travel, Marital Status
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, ['OverTime','BusinessTravel','MaritalStatus']):
    rates = df.groupby(col)['Attrition_Flag'].mean().sort_values(ascending=False)*100
    rates.plot.bar(ax=ax, color='#DC2626', edgecolor='white')
    ax.set_title(f'Attrition by {col}')
    ax.set_ylabel('Rate (%)')
    ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

## 🔧 Step 4: Feature Engineering

In [ ]:
df_fe = df.copy()

df_fe['IncomeToAge']       = df_fe['MonthlyIncome'] / df_fe['Age']
df_fe['CareerStability']   = df_fe['YearsAtCompany'] / (df_fe['TotalWorkingYears'] + 1)
df_fe['PromotionLag']      = df_fe['YearsSinceLastPromotion'] - df_fe['YearsInCurrentRole']
df_fe['SatisfactionScore'] = (df_fe['JobSatisfaction'] + df_fe['EnvironmentSatisfaction'] +
                               df_fe['RelationshipSatisfaction'] + df_fe['WorkLifeBalance']) / 4
df_fe['EngagementScore']   = (df_fe['JobInvolvement'] + df_fe['JobSatisfaction']) / 2
df_fe['IsHighRisk']        = (
    (df_fe['OverTime']=='Yes').astype(int) +
    (df_fe['JobSatisfaction']<=2).astype(int) +
    (df_fe['WorkLifeBalance']==1).astype(int) +
    (df_fe['YearsAtCompany']<2).astype(int)
)

cat_cols = [c for c in df_fe.select_dtypes('object').columns if c != 'Attrition']
for col in cat_cols:
    df_fe[col] = LabelEncoder().fit_transform(df_fe[col])

print(f'Features after engineering: {df_fe.shape[1]}')

## 🤖 Step 5: Model Training & Evaluation

In [ ]:
feature_cols = [c for c in df_fe.columns if c not in ['Attrition','Attrition_Flag']]
X = df_fe[feature_cols]
y = df_fe['Attrition_Flag']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                     random_state=42, stratify=y)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree'      : DecisionTreeClassifier(max_depth=6, random_state=42),
    'Random Forest'      : RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42),
    'Gradient Boosting'  : GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, random_state=42),
    'SVM'               : SVC(probability=True, kernel='rbf', random_state=42),
}

results = {}
for name, model in models.items():
    scaled = name in ['Logistic Regression','SVM']
    Xtr = X_train_sc if scaled else X_train
    Xte = X_test_sc  if scaled else X_test
    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)
    y_prob = model.predict_proba(Xte)[:,1]
    results[name] = dict(model=model, y_pred=y_pred, y_prob=y_prob,
                         auc=roc_auc_score(y_test,y_prob), f1=f1_score(y_test,y_pred))
    print(f'{name:<22} AUC={results[name]["auc"]:.4f}  F1={results[name]["f1"]:.4f}')

In [ ]:
# ROC Curves
fig, ax = plt.subplots(figsize=(9, 7))
colors = ['#2563EB','#16A34A','#DC2626','#D97706','#7C3AED']
for (name, r), c in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, r['y_prob'])
    ax.plot(fpr, tpr, label=f"{name} (AUC={r['auc']:.3f})", color=c, linewidth=2)
ax.plot([0,1],[0,1],'--', color='gray')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix for best model
best_name = max(results, key=lambda k: results[k]['auc'])
best = results[best_name]
print(f'Best Model: {best_name} (AUC={best["auc"]:.4f})')
print(classification_report(y_test, best['y_pred'], target_names=['Stay','Leave']))

fig, ax = plt.subplots(figsize=(6,5))
cm = confusion_matrix(y_test, best['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Stay','Leave'], yticklabels=['Stay','Leave'])
ax.set_title(f'Confusion Matrix — {best_name}')
plt.tight_layout()
plt.show()

## 🏆 Step 6: Feature Importance — Top Attrition Drivers

In [ ]:
rf = results['Random Forest']['model']
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
top20 = importances.head(20)
colors = ['#DC2626']*5 + ['#2563EB']*15
top20[::-1].plot.barh(ax=ax, color=colors[::-1])
ax.set_title('Top 20 Attrition Drivers (Random Forest)')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.show()
print('Top 10 Drivers:')
print(importances.head(10).to_string())

## 💡 Step 7: HR Insights & Recommendations

| # | Finding | Recommendation |
|---|---------|----------------|
| 1 | Overtime workers have ~2× higher attrition | Cap overtime, offer comp-time |
| 2 | Low job satisfaction (1–2) is a strong signal | Quarterly pulse surveys + follow-up |
| 3 | Employees with <2 years tenure are highest risk | Structured onboarding + mentoring |
| 4 | Frequent business travel correlates with leaving | Review travel policy + recovery days |
| 5 | Low monthly income drives attrition | Annual market benchmarking, target P60 |
| 6 | Single employees leave at higher rates | Work-life flexibility programmes |

**Deploy:** Run attrition probability scores monthly → route employees >60% risk to retention interviews.

In [ ]:
# Save the best model
os.makedirs('../models', exist_ok=True)
joblib.dump(rf, '../models/random_forest_model.pkl')
joblib.dump(scaler, '../models/scaler.pkl')
print('✔ Models saved to /models/')